COMPARISON OF CUSTOM FAISS RAG & LANGCHAIN FAISS RAG

In [9]:
###########################
####  CUSTOM FAISS RAG  ###

import faiss
import pandas as pd
import numpy as np
from langchain_mistralai import MistralAIEmbeddings, ChatMistralAI

# Load index
index = faiss.read_index("../data/processed/faiss/reviews.index")

# Load metadata
df = pd.read_csv("../data/processed/faiss/metadata.csv")

embeddings = MistralAIEmbeddings(model="mistral-embed")
llm = ChatMistralAI(model="mistral-small")

def retrieve(query, k=5):

    query_vector = embeddings.embed_query(query)

    # Convert to numpy array for FAISS
    query_vector = np.array([query_vector]).astype("float32")

    distances, indices = index.search(query_vector, k)

    docs = df.iloc[indices[0]]["review_text"].tolist()

    return docs

Custom FAISS RAG pipeline 

Query

 ↓

Embedding

 ↓

FAISS Search

 ↓

Top K Reviews

 ↓

Mistral LLM

 ↓
 
Answer

In [ ]:
###########################
#### LangChain Faiss RAG ##

# Langchain faiss rag manages the retrieval , memory , prompt Orchestration

In [10]:
#Creation of Langchain faiss VectorStore 



import pandas as pd
from langchain_mistralai import MistralAIEmbeddings
from langchain_community.vectorstores import FAISS

# Load metadata
df = pd.read_csv("../data/processed/faiss/metadata.csv")

# Assuming review column name
texts = df["review_text"].tolist()

# Embedding model
embeddings = MistralAIEmbeddings(model="mistral-embed")

# Create vector store
vectorstore = FAISS.from_texts(texts, embeddings)

# Save vector database
vectorstore.save_local("../data/processed/langchain_faiss")

In [11]:
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferWindowMemory

embeddings = MistralAIEmbeddings(model="mistral-embed")

vectorstore = FAISS.load_local(
    "../data/processed/langchain_faiss",
    embeddings,
    allow_dangerous_deserialization=True
)

llm = ChatMistralAI(model="mistral-small")

memory = ConversationBufferWindowMemory(
    memory_key="chat_history",
    return_messages=True,
    k=5
)

chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k":5}),
    memory=memory
)

In [12]:
import time

questions = [
    "What are the most common complaints about kitchen appliances?",
    "Are there delivery delays mentioned in reviews?",
    "What do customers say about DreamRest mattress?",
    "Do customers complain about noise in HomeNest products?",
    "Are there safety concerns reported?"
]

print("\n==============================")
print("CUSTOM FAISS RAG")
print("==============================")

for q in questions:

    start = time.time()

    docs = retrieve(q, k=5)

    context = "\n".join(docs)

    prompt = f"""
    Use the following customer reviews to answer the question.

    Reviews:
    {context}

    Question:
    {q}

    Answer:
    """

    response = llm.invoke(prompt)

    latency = time.time() - start

    print("\nQuestion:", q)
    print("Answer:", response.content)
    print("Latency:", round(latency,2), "seconds")


print("\n\n==============================")
print("LANGCHAIN FAISS RAG")
print("==============================")

for q in questions:

    start = time.time()

    result = chain({"question": q})

    latency = time.time() - start

    print("\nQuestion:", q)
    print("Answer:", result["answer"])
    print("Latency:", round(latency,2), "seconds")


CUSTOM FAISS RAG

Question: What are the most common complaints about kitchen appliances?
Answer: Based on the provided customer reviews, the most common complaints about kitchen appliances are:

1. **Poor Quality** – Many users express dissatisfaction with the overall build or performance, feeling that the product does not meet their expectations despite functioning adequately.
2. **Noise Issues** – Complaints about loud, grinding, or unpleasant noises from appliances (e.g., motor noise in the CrispAir Fryer XL).
3. **Safety Hazards** – Concerns about malfunctioning appliances that pose risks, such as sparking (SliceMaster Food Processor).

These complaints highlight issues related to durability, noise, and safety in kitchen appliances.
Latency: 1.88 seconds

Question: Are there delivery delays mentioned in reviews?
Answer: Yes, there are delivery delays mentioned in the reviews. Specifically:

- One review states that the delivery took 3 weeks for the CloudPillow Set (2-pack) and th

C:\Users\TA26012\AppData\Local\Temp\ipykernel_7900\1085757314.py:52: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  result = chain({"question": q})



Question: What are the most common complaints about kitchen appliances?
Answer: Based on the provided context, the most common complaints about kitchen appliances include:

1. **Noise**: Issues with motor noise, such as the "horrible grinding noise" reported with the CrispAir Fryer XL.
2. **Quality Issues**: Expectations not being met in terms of build quality or performance, as seen with the SteamKing Rice Cooker and the CrispAir Fryer XL.
3. **Safety Hazards**: Problems like sparking when plugged in, as reported with the SliceMaster Food Processor.

These complaints highlight concerns about noise levels, durability, and safety in kitchen appliances.
Latency: 2.21 seconds

Question: Are there delivery delays mentioned in reviews?
Answer: Yes, there are mentions of delivery delays in the reviews provided:

1. "WRONG ITEM DELIVERED INITIALLY. REPLACEMENT MIRRORPRO LED VANITY MIRROR FINALLY CAME AFTER 10 DAY." - This indicates a delay in receiving the correct product, with the replaceme